# KG-GT Method 1 — Preprocessing + Transformer + GAT

Pipeline covered in this notebook:

1. Load config and project helpers.
2. Build train / val / test datasets.
3. Fit CCA and EMG normalization from train only.
4. Build the separate KG-GT model variant.
5. Run a forward-pass smoke test through preprocessing → transformer → GAT.

This notebook stops before full training so the Phase 4 architecture can be checked quickly.

In [ ]:
# @title imports and setup
from pathlib import Path
import sys

import torch
import yaml
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models import build_kg_gt_from_config
from src.pipelines import build_dataset, fit_cca_and_emg_stats, prepare_batch_factory, unique_series_arrays
from src.training import evaluate
from src.utils import get_device, set_seed

set_seed(42)
DEVICE = get_device()
print('ROOT:', ROOT)
print('DEVICE:', DEVICE)


In [ ]:
# @title load configuration
cfg = yaml.safe_load(open(ROOT / 'configs' / 'default.yaml', encoding='utf-8'))

PARTICIPANTS = cfg['data']['participants']
N_CCA = cfg['preprocessing']['cca']['n_components']
BATCH_SIZE = cfg['training']['batch_size']
WINDOW_SIZE = cfg['data']['window_size']
EMG_NAMES = ['Ant. Deltoid', 'Ext. Carpi Rad.', 'Flex. Digitorum', 'Ext. Dig. Comm.', '1st Dors. Inteross.']

print('model type:', cfg['model'].get('type', 'transformer_regressor'))
print('participants:', PARTICIPANTS)
print('window size:', WINDOW_SIZE)


## 1. Build datasets

This keeps the existing preprocessing path intact and reuses the shared dataset builder.

In [ ]:
# @title build_datasets
data_cfg = cfg['data']
train_ds = build_dataset('train', cfg, PARTICIPANTS, ROOT)
val_ds = build_dataset('val', cfg, PARTICIPANTS, ROOT)
test_ds = build_dataset('test', cfg, PARTICIPANTS, ROOT)

print('train windows:', len(train_ds))
print('val windows:', len(val_ds))
print('test windows:', len(test_ds))


## 2. Fit train-only CCA and EMG stats

In [ ]:
# @title unique_series_arrays
cca_gpu, emg_mean, emg_std = fit_cca_and_emg_stats(train_ds, cfg['preprocessing']['cca'], DEVICE)
prepare_batch = prepare_batch_factory(
    cca_projector=cca_gpu,
    emg_mean=emg_mean,
    emg_std=emg_std,
    device=DEVICE,
    n_cca=N_CCA,
)

print('CCA device:', cca_gpu.device)
print('EMG mean:', emg_mean.detach().cpu().tolist())
print('EMG std:', emg_std.detach().cpu().tolist())


## 3. Build KG-GT model

The model remains a separate variant from the current transformer regressor.

In [ ]:
# @title KGGTModel
model_cfg = dict(cfg)
model_cfg['model'] = dict(cfg['model'])
model_cfg['model']['type'] = 'kg_gt'
model = build_kg_gt_from_config(model_cfg, input_dim=N_CCA, kin_dim=cfg['data'].get('n_kin_features', 13)).to(DEVICE)
print(model)


## 4. Smoke test the full preprocessing → transformer → GAT path

In [ ]:
# @title forward_smoke_test
sample_eeg, sample_kin, sample_emg = test_ds[0]
model_inputs, target = prepare_batch(sample_eeg.unsqueeze(0), sample_kin.unsqueeze(0), sample_emg.unsqueeze(0))
with torch.no_grad():
    pred = model(**model_inputs)

print('pred shape:', tuple(pred.shape))
print('target shape:', tuple(target.shape))


## 5. Optional evaluation hook

Use this only when you want a slow full test-set pass.

In [ ]:
# @title optional_evaluation
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE.type == 'cuda'))
# metrics = evaluate(model, test_loader, prepare_batch, DEVICE, channel_names=EMG_NAMES, n_channels=5)
# print(metrics.as_table())
